# Kendi sesin — Hızlı Okuma için ses eğitimi

Bu not defteri senin sesinden, Hızlı Okuma'nın telefonda **internetsiz** okuyabileceği küçük bir ses modeli (Piper) eğitir.

**Nasıl çalışır?**
1. 10–30 saniyelik bir kaydını yüklersin ya da burada kaydedersin.
2. Chatterbox (açık kaynak ses klonlama) bu kayıttan yola çıkıp senin sesinle ~1.000 Türkçe cümle okur.
3. Whisper bu kayıtları dinler, yanlış okunanları ayıklar.
4. Türkçe bir Piper sesi (dfki) senin kayıtlarınla ince ayarlanır: ses Türkçeyi baştan öğrenmez, yalnızca senin sesine uyar.
5. Sonunda Google Drive'ına iki dosya kaydedilir (`.onnx` ve `.onnx.json`). Telefona indirip Hızlı Okuma → Ayarlar → Ses → **Ses paketi ekle** ile eklersin.

**Gerekenler:** Google hesabı ve **ekran kartlı çalışma zamanı** (menü: *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*). Ücretsiz Colab'da toplam 4–6 saat sürer. Oturum koparsa her adım kaldığı yerden devam eder: her şey Drive'da. Drive'da ~5 GB boş yer gerekir.

**Önemli:** Yalnızca kendi sesini ya da sahibinden açık izin aldığın bir sesi kullan. Başlangıç noktası olan dfki verisi CC BY-NC-SA 4.0 lisanslı olduğu için ortaya çıkan ses **kişisel, ticari olmayan** kullanım içindir.

Hücreleri yukarıdan aşağıya sırayla çalıştır (her hücrenin solundaki ▶). Adım adım anlatım: `voice-training/README.md`.

In [ ]:
#@title 1. Ayarlar
SES_ADI = "Benim sesim"  #@param {type:"string"}
CUMLE_SAYISI = 1000  #@param {type:"slider", min:200, max:1300, step:100}
EPOCH = 300  #@param {type:"slider", min:50, max:1000, step:50}
#@markdown Cümle arttıkça ses iyileşir ama üretim uzar (1.000 cümle ≈ 65 dk ses, T4'te ~2 saat). Epoch: eğitim turu; 300 tur T4'te ~2–3 saat.
SES_BENIM = False  #@param {type:"boolean"}
#@markdown ☝️ **Bu ses benim ya da sahibinden açık izin aldım.** İşaretlenmeden devam edilmez.
DRIVE_KLASORU = "HizliOkumaSes"  #@param {type:"string"}
DAL = "claude/speed-reading-app-plan-kzot5o"
assert SES_BENIM, "Devam etmek için kutuyu işaretle: yalnızca kendi sesin ya da izinli bir ses klonlanabilir."
print("Tamam.")

In [ ]:
#@title 2. Hazırlık: ekran kartı, Drive, eğitim dosyaları
import pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("Ekran kartı yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç ve baştan çalıştır.")
print("Ekran kartı:", gpu.stdout.strip())

from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive") / DRIVE_KLASORU
for sub in ("kayit", "wavs", "yedek", "paket", "temel"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

REPO = pathlib.Path("/content/hizliokuma")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", DAL,
                    "https://github.com/mustafa2049/Notebooklm.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
TOOLS = REPO / "voice-training"
CORPUS = TOOLS / "corpus.tsv"
WORK = pathlib.Path("/content/egitim")
print("Drive klasörü:", DRIVE)

## 3. Sesini kaydet (10–30 saniye)

Sessiz bir odada, mikrofona 20–30 cm uzaktan, **doğal hızınla** şu paragrafı oku:

> Okumak, insanın kendi zihninde çıktığı uzun bir yolculuktur. Her sayfada yeni bir kapı açılır; bu kapılardan geçtikçe dünyaya başka gözlerle bakmaya başlarız. Bugün hava güzel, ama yarın yağmur yağabilir. Sen de benimle gelir misin?

İki yol var: **3a** ile burada kaydet ya da telefonunla kaydettiğin dosyayı (wav, mp3, m4a) **3b** ile yükle. Biri yeterli. Kayıt ne kadar temizse (yankısız, gürültüsüz) ses o kadar iyi olur.

In [ ]:
#@title 3a. Burada kaydet (tarayıcının mikrofonu)
SURE_SANIYE = 25  #@param {type:"slider", min:10, max:40, step:5}
from base64 import b64decode
from google.colab import output
from IPython.display import Audio, Javascript, display

display(Javascript("""
async function kaydet(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const parts = [];
  recorder.ondataavailable = (e) => parts.push(e.data);
  const note = document.createElement('div');
  note.style.fontSize = '18px';
  note.textContent = '🔴 Kayıt başladı — paragrafı oku';
  document.body.appendChild(note);
  recorder.start();
  await new Promise((r) => setTimeout(r, ms));
  recorder.stop();
  await new Promise((r) => (recorder.onstop = r));
  stream.getTracks().forEach((t) => t.stop());
  note.textContent = '✅ Kayıt bitti';
  const bytes = new Uint8Array(await new Blob(parts).arrayBuffer());
  let binary = '';
  for (let i = 0; i < bytes.length; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
"""))
ham = DRIVE / "kayit" / "kayit.webm"
ham.write_bytes(b64decode(output.eval_js(f"kaydet({SURE_SANIYE * 1000})")))
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(ham), "-ac", "1", "-ar", "24000",
                str(DRIVE / "kayit" / "kayit.wav")], check=True)
display(Audio(str(DRIVE / "kayit" / "kayit.wav")))
print("Kaydedildi. Dinle; beğenmezsen hücreyi yeniden çalıştır.")

In [ ]:
#@title 3b. Ya da kaydını yükle
from google.colab import files
from IPython.display import Audio, display
for ad, icerik in files.upload().items():
    ham = DRIVE / "kayit" / ad
    ham.write_bytes(icerik)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(ham), "-ac", "1", "-ar", "24000",
                    str(DRIVE / "kayit" / "kayit.wav")], check=True)
sure = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                             str(DRIVE / "kayit" / "kayit.wav")], capture_output=True, text=True).stdout or 0)
print(f"Kayıt {sure:.0f} saniye" + ("" if 8 <= sure <= 45 else " — 10–30 saniye en iyisi"))
display(Audio(str(DRIVE / "kayit" / "kayit.wav")))

In [ ]:
#@title 4. Araçları kur (5–10 dk; oturum başına bir kez)
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
ENVS = pathlib.Path("/content/envs")

def ortam(ad, *paketler):
    """Her araç kendi Python 3.11 ortamında: Colab'ın paketleriyle çakışmasın."""
    yol = ENVS / ad
    if not (yol / "bin" / "python").exists():
        subprocess.run(["uv", "venv", "-q", "--python", "3.11", str(yol)], check=True)
    subprocess.run(["uv", "pip", "install", "-q", "--python", str(yol / "bin" / "python"), *paketler], check=True)
    return str(yol / "bin" / "python")

PY_KLON = ortam("klon", "chatterbox-tts==0.1.7", "setuptools<81")
# Whisper'ın ekran kartında çalışması için cuBLAS ve cuDNN (yoksa işlemciyle çalışır)
PY_DENETIM = ortam("denetim", "faster-whisper==1.2.1", "soundfile", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*")
PY_PIPER = ortam("piper", "piper-tts[train]==1.8.0", "torch==2.8.0")

# Piper eğitiminin hızlandırıcı parçası (Cython) pakette derlenmiş gelmiyor:
# aynı sürümün kaynağından derlenir
piper_dir = subprocess.run([PY_PIPER, "-c", "import piper, os; print(os.path.dirname(piper.__file__))"],
                           capture_output=True, text=True, check=True).stdout.strip()
ma = pathlib.Path(piper_dir) / "train" / "vits" / "monotonic_align"
if not list((ma / "monotonic_align").glob("core*.so")):
    subprocess.run(["curl", "-sL", "-o", str(ma / "core.pyx"),
                    "https://raw.githubusercontent.com/OHF-Voice/piper1-gpl/v1.8.0/src/piper/train/vits/monotonic_align/core.pyx"], check=True)
    subprocess.run([str(pathlib.Path(PY_PIPER).parent / "cythonize"), "-q", "-i", "core.pyx"], cwd=ma, check=True)
    (ma / "monotonic_align").mkdir(exist_ok=True)
    for so in ma.glob("core*.so"):
        so.rename(ma / "monotonic_align" / so.name)
subprocess.run([PY_PIPER, "-c", "from piper.train.vits.monotonic_align import maximum_path"], check=True)
print("Araçlar hazır.")

In [ ]:
#@title 5. Cümleleri senin sesinle üret (en uzun adım; kaldığı yerden devam eder)
assert (DRIVE / "kayit" / "kayit.wav").exists(), "Önce 3a ya da 3b ile kaydını ekle."
!{PY_KLON} {TOOLS}/generate.py --corpus {CORPUS} --ref {DRIVE}/kayit/kayit.wav --out {DRIVE}/wavs --limit {CUMLE_SAYISI}

In [ ]:
#@title 6. Birkaç cümleyi dinle
import random
from IPython.display import Audio, display
kayitlar = sorted((DRIVE / "wavs").glob("c*.wav"))
print(len(kayitlar), "cümle hazır")
for yol in random.sample(kayitlar, min(3, len(kayitlar))):
    print(yol.stem)
    display(Audio(str(yol)))

In [ ]:
#@title 7. Yanlış okunanları ayıkla (Whisper; 10–20 dk)
LIBS = subprocess.run([PY_DENETIM, "-c", "import nvidia.cublas.lib as a, nvidia.cudnn.lib as b; print(list(a.__path__)[0] + ':' + list(b.__path__)[0])"],
                      capture_output=True, text=True).stdout.strip()
!LD_LIBRARY_PATH={LIBS} {PY_DENETIM} {TOOLS}/filter.py --corpus {CORPUS} --wavs {DRIVE}/wavs --out {DRIVE}/metadata.csv

In [ ]:
#@title 8. Türkçe başlangıç modelini indir (846 MB; bir kez)
temel = DRIVE / "temel" / "dfki.ckpt"
if not temel.exists():
    subprocess.run(["curl", "-sL", "-o", f"{temel}.part",
                    "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/tr/tr_TR/dfki/medium/epoch%3D5679-step%3D1489110.ckpt"], check=True)
    pathlib.Path(f"{temel}.part").rename(temel)
print(temel, round(temel.stat().st_size / 1e6), "MB")

## 9. Eğit

Kontrol noktası 10 dakikada bir Drive'a (`yedek`) kopyalanır; oturum koparsa bu hücreyi yeniden çalıştırınca kaldığı yerden sürer.

**Ara ara dinlemek için:** hücreyi durdur (■), 10. adımı çalıştır, sonra bu hücreyi yeniden başlat. Ses 100–200 epoch'tan sonra sana benzemeye başlar. Ekran kartı belleği yetmezse (`CUDA out of memory`) `BATCH`'i 8 yap.

In [ ]:
#@title 9. Eğit (kaldığı yerden devam eder)
BATCH = 16  #@param {type:"integer"}
!{PY_PIPER} {TOOLS}/train.py --metadata {DRIVE}/metadata.csv --wavs {DRIVE}/wavs --work {WORK} --base {DRIVE}/temel/dfki.ckpt --epochs {EPOCH} --batch-size {BATCH} --name "{SES_ADI}" --backup {DRIVE}/yedek

In [ ]:
#@title 10. Paketi hazırla ve dinle
!{PY_PIPER} {TOOLS}/export.py --work {WORK} --backup {DRIVE}/yedek --name "{SES_ADI}" --out {DRIVE}/paket --corpus {CORPUS}
from IPython.display import Audio, display
display(Audio(str(DRIVE / "paket" / "ornek.wav")))

## 11. Telefona aktar

1. Telefonda Google Drive'ı aç → `HizliOkumaSes/paket` klasörü.
2. `tr_TR-…-medium.onnx` ve `tr_TR-…-medium.onnx.json` dosyalarını indir (⋮ → İndir).
3. Hızlı Okuma → Ayarlar → Ses → **Ses paketi ekle** → iki dosyayı birlikte seç.
4. **Dene** ile dinle. Okuyucudaki kulaklık düğmesi ve "Kelimeyi dinle" artık senin sesinle okur — internetsiz.

Web sürümünde (tarayıcıda) zip dosyasını da seçebilirsin.

**Ses beğendiğin gibi değil mi?** Eğitimi uzat (1. adımda EPOCH'u artır, 9. adımı yeniden çalıştır; kaldığı yerden sürer) ya da daha temiz bir kayıtla 3. adımdan başla (önce Drive'daki `wavs` klasörünü sil).